In [11]:
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

In [12]:
#load data
X_train = np.load("../data/processed/X_train_risk.npy")
X_test = np.load("../data/processed/X_test_risk.npy")
y_train = np.load("../data/processed/y_train_risk.npy")
y_test = np.load("../data/processed/y_test_risk.npy")

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (170152, 26)
Test: (42539, 26)


In [13]:
#Apply smote to balance the classes
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before:", np.bincount(y_train))
print("After:", np.bincount(y_train_resampled))

Before: [25505 86695 57952]
After: [86695 86695 86695]


In [14]:
#build model
input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(3, activation='softmax')
])

In [15]:
#complete model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
classes = np.unique(y_train_resampled)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_resampled
)

class_weights = dict(zip(classes, class_weights))

print(class_weights)

{0: 1.0, 1: 1.0, 2: 1.0}


In [17]:
#train model
history = model.fit(
    X_train_resampled,
    y_train_resampled,
    validation_split=0.2,
    epochs=30,
    batch_size=128,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/30
1626/1626 [==============================] - 7s 4ms/step - loss: 0.5330 - accuracy: 0.6959 - val_loss: 0.4852 - val_accuracy: 0.4474
Epoch 2/30
1626/1626 [==============================] - 6s 4ms/step - loss: 0.4706 - accuracy: 0.7210 - val_loss: 0.5112 - val_accuracy: 0.4474
Epoch 3/30
1626/1626 [==============================] - 6s 3ms/step - loss: 0.4698 - accuracy: 0.7212 - val_loss: 0.4969 - val_accuracy: 0.4474
Epoch 4/30
1626/1626 [==============================] - 6s 3ms/step - loss: 0.4692 - accuracy: 0.7214 - val_loss: 0.5291 - val_accuracy: 0.4474
Epoch 5/30
1626/1626 [==============================] - 6s 4ms/step - loss: 0.4691 - accuracy: 0.7214 - val_loss: 0.4743 - val_accuracy: 0.4474
Epoch 6/30
1626/1626 [==============================] - 6s 4ms/step - loss: 0.4691 - accuracy: 0.7214 - val_loss: 0.5298 - val_accuracy: 0.4474
Epoch 7/30
1626/1626 [==============================] - 5s 3ms/step - loss: 0.4686 - accuracy: 0.7214 - val_loss: 0.4973 - val_accuracy:

In [18]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred_classes))
print(classification_report(y_test, y_pred_classes))

1330/1330 [==============================] - 1s 979us/step
Accuracy: 0.6603587296363337
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6398
           1       0.60      1.00      0.75     21693
           2       0.00      0.00      0.00     14448

    accuracy                           0.66     42539
   macro avg       0.53      0.67      0.58     42539
weighted avg       0.46      0.66      0.53     42539



d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [20]:
#save results
import json

report = classification_report(y_test, y_pred_classes, output_dict=True)

with open("../models/dl_classification_report.json", "w") as f:
    json.dump(report, f)

d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [21]:
#save model
model.save("../models/deep_learning_model.h5")
print("Model saved")

Model saved


d:\Accademic\Research\Project\ThyroidCare\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
